# IDE Configuration — MCP Servers + Self-Hosted Model

This notebook configures Cursor, VS Code, and Claude Code to work as a **coding assistant** using:
- **MCP tool servers** deployed on OpenShift (5 servers)
- **Self-hosted model** (qwen36-27b) served via RHOAI MaaS gateway

No external AI API keys required — everything runs on your cluster.

**Sections:**
1. Discover MCP server endpoints
2. Cursor IDE config (MCP + Model)
3. VS Code config (MCP + Model)
4. Claude Code config (MCP + Model)
5. MaaS Gateway mode (Phase 3 — unified endpoint with auth)
6. Verify connectivity

## 0. Load Environment

In [ ]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

# Discover MCP server routes
routes_result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

mcp_urls = {}
for line in routes_result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        short_name = name.replace("mcp-", "")
        mcp_urls[short_name] = f"https://{host}/mcp"

print(f"Cluster:       {CLUSTER_DOMAIN}")
print(f"Model:         {MODEL_NAME}")
print(f"Model URL:     {MODEL_ENDPOINT}/v1")
print(f"MaaS API Key:  {'set' if MAAS_API_KEY else 'not set (direct access)'}")
print(f"MCP Servers:   {len(mcp_urls)} discovered")
for name, url in sorted(mcp_urls.items()):
    print(f"  {name:<20} {url}")

---
## 1. Cursor IDE

### MCP Configuration

Create `.cursor/mcp.json` in your project root:

In [ ]:
cursor_mcp = {"mcpServers": {}}
for name, url in sorted(mcp_urls.items()):
    cursor_mcp["mcpServers"][name] = {"url": url}

print("=== .cursor/mcp.json ===")
print(json.dumps(cursor_mcp, indent=2))

### Model Configuration (Cursor)

In Cursor: **Settings → Models → Add Model**

| Setting | Value |
|---------|-------|
| Provider | OpenAI Compatible |
| Base URL | `<MODEL_ENDPOINT>/v1` |
| Model Name | `<MODEL_NAME>` |
| API Key | Empty (direct) or MaaS API key (Phase 3) |

In [ ]:
print("Cursor Model Settings:")
print("=" * 50)
print(f"  Provider:   OpenAI Compatible")
print(f"  Base URL:   {MODEL_ENDPOINT}/v1")
print(f"  Model Name: {MODEL_NAME}")
print(f"  API Key:    {MAAS_API_KEY if MAAS_API_KEY else '(leave empty for direct access)'}")
print("")
print("Steps:")
print("  1. Open Cursor Settings (Cmd+,)")
print("  2. Go to Models section")
print("  3. Click 'Add Model' → 'OpenAI Compatible'")
print("  4. Enter the values above")
print(f"  5. Select '{MODEL_NAME}' as your default model")

---
## 2. VS Code (Agent Mode)

### MCP Configuration

Create `.vscode/mcp.json` in your project root (requires VS Code 1.100+):

In [ ]:
vscode_mcp = {"servers": {}}
for name, url in sorted(mcp_urls.items()):
    vscode_mcp["servers"][name] = {"type": "streamableHttp", "url": url}

print("=== .vscode/mcp.json ===")
print(json.dumps(vscode_mcp, indent=2))

### Model Configuration (VS Code)

Add to `.vscode/settings.json` to configure the self-hosted model as a chat provider:

In [ ]:
vscode_model_config = {
    "chat.models": [
        {
            "family": "openai",
            "id": MODEL_NAME,
            "name": f"{MODEL_NAME} (RHOAI)",
            "url": f"{MODEL_ENDPOINT}/v1",
            "apiKey": MAAS_API_KEY if MAAS_API_KEY else "dummy",
            "isDefault": True
        }
    ]
}

print("=== Add to .vscode/settings.json ===")
print(json.dumps(vscode_model_config, indent=2))
print("")
print("Note: VS Code Agent Mode will use this model for code generation and chat.")
if not MAAS_API_KEY:
    print("  apiKey is set to 'dummy' — some providers require a non-empty value.")

---
## 3. Claude Code

### MCP Configuration

Create `.mcp.json` in project root (or `~/.claude/mcp.json` for global):

In [ ]:
claude_mcp = {"mcpServers": {}}
for name, url in sorted(mcp_urls.items()):
    claude_mcp["mcpServers"][name] = {"type": "url", "url": url}

print("=== .mcp.json (Claude Code) ===")
print(json.dumps(claude_mcp, indent=2))

### Model Configuration (Claude Code — Self-Hosted)

Claude Code can connect to **self-hosted models** (not external Claude) via environment variables.

| Env Var | Purpose |
|---------|--------|
| `ANTHROPIC_BASE_URL` | MaaS inference endpoint for the model |
| `ANTHROPIC_AUTH_TOKEN` | MaaS API key (**NOT** `ANTHROPIC_API_KEY`) |
| `ANTHROPIC_DEFAULT_OPUS_MODEL` | Model name for Opus tier |
| `ANTHROPIC_DEFAULT_SONNET_MODEL` | Model name for Sonnet tier |
| `ANTHROPIC_DEFAULT_HAIKU_MODEL` | Model name for Haiku tier |
| `MAX_THINKING_TOKENS` | `0` — disable thinking (vLLM doesn't support it) |
| `CLAUDE_CODE_MAX_OUTPUT_TOKENS` | Max tokens per response |
| `CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS` | Max tokens for file reads |

> **Critical:** Use `ANTHROPIC_AUTH_TOKEN`, not `ANTHROPIC_API_KEY`. The latter does not pass authentication to MaaS correctly.

> Since we have a single model deployed, all three `DEFAULT_MODEL` vars point to the same model name.

In [ ]:
auth_token = MAAS_API_KEY if MAAS_API_KEY else "sk-oai-YOUR-MAAS-API-KEY"

claude_launch_cmd = f"""ANTHROPIC_BASE_URL="{MODEL_ENDPOINT}" \\
ANTHROPIC_AUTH_TOKEN="{auth_token}" \\
ANTHROPIC_DEFAULT_OPUS_MODEL="{MODEL_NAME}" \\
ANTHROPIC_DEFAULT_SONNET_MODEL="{MODEL_NAME}" \\
ANTHROPIC_DEFAULT_HAIKU_MODEL="{MODEL_NAME}" \\
CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000" \\
CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000" \\
MAX_THINKING_TOKENS="0" \\
claude"""

print("=== Launch Claude Code with self-hosted model ===")
print("")
print(claude_launch_cmd)
print("")
print("Copy-paste the command above into your terminal to start Claude Code")
print(f"connected to your self-hosted {MODEL_NAME} model.")
if not MAAS_API_KEY:
    print("")
    print("NOTE: Replace 'sk-oai-YOUR-MAAS-API-KEY' with your actual MaaS API key.")
    print("Generate one via: ../3_maas/2_enable_maas.ipynb (Step 4)")

### Shell Script (Optional)

For convenience, save as `run-claude.sh` in your project:

In [ ]:
script_content = f"""#!/bin/bash
# Launch Claude Code with self-hosted RHOAI model
# Usage: source run-claude.sh

export ANTHROPIC_BASE_URL="{MODEL_ENDPOINT}"
export ANTHROPIC_AUTH_TOKEN="${{MAAS_API_KEY:-{auth_token}}}"
export ANTHROPIC_DEFAULT_OPUS_MODEL="{MODEL_NAME}"
export ANTHROPIC_DEFAULT_SONNET_MODEL="{MODEL_NAME}"
export ANTHROPIC_DEFAULT_HAIKU_MODEL="{MODEL_NAME}"
export CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000"
export CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000"
export MAX_THINKING_TOKENS="0"

echo "Claude Code configured for: ${{ANTHROPIC_DEFAULT_SONNET_MODEL}}"
echo "Endpoint: ${{ANTHROPIC_BASE_URL}}"
claude "$@"
"""

print("=== run-claude.sh ===")
print(script_content)

---
## 4. MaaS Gateway Mode (Phase 3)

After completing Phase 3 (`../3_maas/2_enable_maas.ipynb`), you can switch to the **MaaS Gateway** for unified auth on both model and MCP access.

### Comparison: Direct Route vs MaaS Gateway

| | Direct Route (Phase 1-2) | MaaS Gateway (Phase 3+) |
|-|--------------------------|-------------------------|
| MCP endpoints | 5 separate URLs | 1 unified URL |
| Model endpoint | Direct MaaS API URL | Same (via gateway) |
| Authentication | None | API key (single key for all) |
| Rate limiting | None | Per-subscription token limits |
| Config entries | 5 MCP + 1 model | 1 MCP gateway + 1 model |

In [ ]:
MAAS_GW = f"https://maas-api.{CLUSTER_DOMAIN}"
api_key_display = MAAS_API_KEY if MAAS_API_KEY else "sk-oai-YOUR-KEY"

print("=== MaaS Gateway Mode ===")
print("")
print("Cursor (.cursor/mcp.json):")
cursor_maas = {
    "mcpServers": {
        "mcp-gateway": {
            "url": f"{MAAS_GW}/mcp/mcp",
            "headers": {"Authorization": f"Bearer {api_key_display}"}
        }
    }
}
print(json.dumps(cursor_maas, indent=2))

print("")
print("VS Code (.vscode/mcp.json):")
vscode_maas = {
    "servers": {
        "mcp-gateway": {
            "type": "streamableHttp",
            "url": f"{MAAS_GW}/mcp/mcp",
            "headers": {"Authorization": f"Bearer {api_key_display}"}
        }
    }
}
print(json.dumps(vscode_maas, indent=2))

print("")
print("Claude Code (CLI):")
print(f"  claude mcp add mcp-gateway \\")
print(f"    --transport streamable-http \\")
print(f"    --url \"{MAAS_GW}/mcp/mcp\" \\")
print(f"    --header \"Authorization: Bearer {api_key_display}\"")

---
## 5. Verify Connectivity

In [ ]:
import urllib.request, ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print("=== MCP Server Health Check ===")
print("")

init_payload = json.dumps({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "healthcheck", "version": "1.0"}}
}).encode()

for name, url in sorted(mcp_urls.items()):
    try:
        req = urllib.request.Request(url, data=init_payload,
            headers={"Content-Type": "application/json", "Accept": "application/json, text/event-stream"},
            method="POST")
        with urllib.request.urlopen(req, context=ctx, timeout=5) as resp:
            print(f"  [PASS    ] {name:<20} HTTP {resp.status}")
    except urllib.error.HTTPError as e:
        if e.code == 405:
            print(f"  [PASS    ] {name:<20} HTTP 405 (server active)")
        else:
            print(f"  [FAIL    ] {name:<20} HTTP {e.code}")
    except Exception as e:
        print(f"  [FAIL    ] {name:<20} {str(e)[:40]}")

print("")
print("=== Model Endpoint Check ===")
print("")
try:
    model_url = f"{MODEL_ENDPOINT}/v1/models"
    headers = {"Content-Type": "application/json"}
    if MAAS_API_KEY:
        headers["Authorization"] = f"Bearer {MAAS_API_KEY}"
    req = urllib.request.Request(model_url, headers=headers)
    with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
        data = json.loads(resp.read())
        models = [m["id"] for m in data.get("data", [])]
        print(f"  [PASS    ] Model endpoint reachable")
        print(f"             Available models: {models}")
except urllib.error.HTTPError as e:
    print(f"  [FAIL    ] HTTP {e.code} — check model deployment or API key")
except Exception as e:
    print(f"  [FAIL    ] {e}")

print("")
print("IDE verification commands:")
print("  Cursor:      Settings > MCP > verify green indicators")
print("  VS Code:     Command Palette > 'MCP: List Servers'")
print("  Claude Code: /mcp to list connected servers")

---
## Summary

| IDE | MCP Config | Model Config | Launch |
|-----|-----------|-------------|--------|
| **Cursor** | `.cursor/mcp.json` | Settings → Models → OpenAI Compatible | Normal launch |
| **VS Code** | `.vscode/mcp.json` | `.vscode/settings.json` (`chat.models`) | Normal launch |
| **Claude Code** | `.mcp.json` | Env vars (`ANTHROPIC_BASE_URL` + `ANTHROPIC_AUTH_TOKEN`) | `source run-claude.sh` |

### Key Points

- All IDEs connect to the **same self-hosted model** — no external API keys needed
- MCP tools provide the "hands" (code search, execution, docs) while the model provides the "brain"
- Claude Code uses `ANTHROPIC_AUTH_TOKEN` (not `ANTHROPIC_API_KEY`) for MaaS authentication
- After Phase 3 (MaaS), switch to gateway mode for unified auth + rate limiting

## Next Steps

- `2_run_public_coding_assistant.ipynb` — Run with all 5 MCP tools (internet required)
- `3_run_closed_coding_assistant.ipynb` — Run with 3 local tools only (air-gapped)
- `../3_maas/2_enable_maas.ipynb` — Enable MaaS for production API key management